[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/soumyaroy799/SUIT_solar_workshops/blob/main/workshops/USO_Vikram_2026_Oct/notebooks/lcurve.ipynb)

In [1]:
import sys
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

REPO_URL = "https://github.com/soumyaroy799/SUIT_solar_workshops.git"

if IN_COLAB:

    ROOT = Path("SUIT_solar_workshops")

    if not ROOT.exists():

        print("Cloning workshop repository...")

        subprocess.run(
            ["git", "clone", REPO_URL],
            check=True,
        )

    ROOT = ROOT.resolve()

else:

    ROOT = Path.cwd().resolve()

    while ROOT.name != "SUIT_solar_workshops":

        if ROOT.parent == ROOT:

            raise RuntimeError(
                "Please run inside "
                "SUIT_solar_workshops"
            )

        ROOT = ROOT.parent

import sys

sys.path.insert(
    0,
    str(ROOT),
)

In [2]:
from workshop_setup import setup_workshop

WORKSHOP_DIR, DATA_DIR, DATA_LCURVE_DIR = setup_workshop(ROOT)

Qt backend enabled
Workshop data already available (19 FITS files found).
Light-curve data already available (816 FITS files found).
Point 1: (260.1, 217.8)
Point 2: (509.2, 39.5)

ROI selected.
Run create_submap()
Point 1: (273.8, 212.1)
Point 2: (430.4, 96.6)

ROI selected.
Run create_submap()


In [3]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from glob import glob
from sunpy.map import Map
from datetime import datetime

import astropy.units as u
from astropy.coordinates import SkyCoord
from matplotlib.path import Path
from sunpy import timeseries as ts
from tqdm import *
import matplotlib.patches as patches
import matplotlib.dates as mdates


from sunkit_image.coalignment import calculate_match_template_shift as cal_shift
from sunkit_image.coalignment import apply_shifts 

from matplotlib.lines import Line2D
from matplotlib.colors import PowerNorm
import csv
from sunpy import timeseries as ts

import warnings
warnings.filterwarnings('ignore')

#### Next up we have a couple of functions that will help us along the way. The next function exposure normalises the data (new data = old data/exposure) and changes the exposure time in header too, as well as the same RoI selection snippet we used in the previous code.


In [4]:
def exp_norm(map_sequence):
    mod_rois = []
    for smap in tqdm(map_sequence):
        esf = (smap.meta['cmd_expt'])/1e3 #exposure scaling factor. The header exp. time is saved in ms
        data = smap.data/esf   
        data[data<0] = 0
        smap.meta['cmd_expt'] = 1000 #changing the exposure to 1 sec
        mod_rois.append(Map(data,smap.meta))
    mod_rois = Map(mod_rois,sequence=True)
    return mod_rois

def start_roi_selection(sunpy_map):
    """
    Display a map and collect two mouse clicks.

    Returns
    -------
    fig, ax, roi
    """

    roi = {
        "clicks": [],
    }

    fig = plt.figure(figsize=(8, 8))
    ax = fig.add_subplot(projection=sunpy_map)

    sunpy_map.plot(axes=ax)

    ax.set_title(
        "Click two opposite corners of the ROI"
    )

    def onclick(event):

        if (
            event.xdata is None
            or event.ydata is None
        ):
            return

        roi["clicks"].append(
            (event.xdata, event.ydata)
        )

        print(
            f"Point {len(roi['clicks'])}: "
            f"({event.xdata:.1f}, "
            f"{event.ydata:.1f})"
        )

        if len(roi["clicks"]) == 2:

            (x1, y1), (x2, y2) = roi["clicks"]

            xmin = min(x1, x2)
            xmax = max(x1, x2)

            ymin = min(y1, y2)
            ymax = max(y1, y2)

            rect = patches.Rectangle(
                (xmin, ymin),
                xmax - xmin,
                ymax - ymin,
                fill=False,
                linewidth=2,
                color="red",
            )

            ax.add_patch(rect)

            fig.canvas.draw_idle()

            print(
                "\nROI selected."
                "\nRun create_submap()"
            )

    fig.canvas.mpl_connect(
        "button_press_event",
        onclick,
    )

    plt.show()

    return fig, ax, roi

def create_submap(
    sunpy_map,
    roi,
):
    """
    Create a submap from ROI clicks.
    """

    if len(roi["clicks"]) != 2:

        raise RuntimeError(
            "ROI not yet selected."
        )

    (x1, y1), (x2, y2) = roi["clicks"]

    xmin = min(x1, x2)
    xmax = max(x1, x2)

    ymin = min(y1, y2)
    ymax = max(y1, y2)

    bottom_left = u.Quantity(
        [xmin, ymin],
        u.pixel,
    )

    top_right = u.Quantity(
        [xmax, ymax],
        u.pixel,
    )

    return sunpy_map.submap(
        bottom_left=bottom_left,
        top_right=top_right,
    )

def co_align(map_sequence,template,layer_index=0,clip=False):
    
    shifts = cal_shift(map_sequence,layer_index=layer_index,template=template)

    plate_scale = map_sequence[0].scale[0]
    dx = -shifts["x"] / plate_scale
    dy = -shifts["y"] / plate_scale
    coaligned_maps = apply_shifts(map_sequence, xshift=dx, yshift=dy,clip=clip)
    return coaligned_maps

def make_lc(mask,map_sequence):
    lc=[];time=[]
    for smap in map_sequence:
        data = smap.data
        lc.append( np.sum(data[mask]) )
        time.append(datetime.strptime(smap.meta['DHOBT_DT'].split('.')[0],"%Y-%m-%dT%H:%M:%S")) 
    return time,lc

# crops the maps in the mapsequence using the corners of the sample map provided.

def crop_maps(mc,sample):
    z=[]
    b_left =  sample.bottom_left_coord
    t_right = sample.top_right_coord
    x1,y1 = sample.world_to_pixel(b_left)
    x2,y2 = sample.world_to_pixel(t_right)
    for temp in mc:
        z.append(temp.submap(bottom_left = [x1,y1]*u.pix,top_right = [x2,y2]*u.pix))  
    z=Map(z,sequence=True)
    return z

def align(mc, index=0):

    # Select two opposite corners of the ROI
    fig, ax, roi = start_roi_selection(mc[index])

    # Wait for ROI selection to finish
    while len(roi["clicks"]) < 2:
        plt.pause(0.1)

    plt.close(fig)

    # Create the SunPy template from the selected ROI
    template = create_submap(mc[index], roi)

    # Display the template and wait for confirmation
    fig, ax = plt.subplots(
        figsize=(8, 8),
        subplot_kw={"projection": template}
    )
    template.plot(axes=ax)
    ax.set_title(
        "Click anywhere to close the template and begin coalignment",
        fontweight="bold",
        color="red"
    )
    plt.waitforbuttonpress()
    plt.close(fig)

    # Align the sequence
    aligned = co_align(
        map_sequence=mc,
        layer_index=index,
        template=template,
        clip=False
    )

    aligned.peek()
    plt.show()

    return aligned

### We start by loading data for different filters as sunpy maps.

In [5]:
smaps = {}

# 1. Load NB03 first to establish the reference map
files3 = sorted(
    (DATA_LCURVE_DIR / "nb03").glob("*NB03.fits")
)

if not files3:
    raise FileNotFoundError("No NB03 FITS files found.")

maps3 = exp_norm(Map(files3, sequence=True))
smaps["nb03"] = crop_maps(maps3, maps3[0])

reference_map = smaps["nb03"][0]

print(f"Loaded NB03: {len(files3)} maps")

# 2. Load the remaining filters using NB03 as the reference
for i in range(1, 9):
    if i == 3:
        continue

    nb = f"nb{i:02d}"

    files = sorted(
        (DATA_LCURVE_DIR / nb).glob(f"*NB{i:02d}.fits")
    )

    if not files:
        raise FileNotFoundError(f"No files found for {nb.upper()}")

    maps = exp_norm(Map(files, sequence=True))
    smaps[nb] = crop_maps(maps, reference_map)

    print(f"Loaded {nb.upper()}: {len(files)} maps")

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████| 51/51 [00:00<00:00, 189.45it/s]


Loaded NB03: 51 maps


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████| 51/51 [00:00<00:00, 202.54it/s]


Loaded NB01: 51 maps


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████| 51/51 [00:00<00:00, 206.77it/s]


...

Loaded NB06: 51 maps


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████| 51/51 [00:00<00:00, 202.64it/s]


Loaded NB07: 51 maps


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████| 51/51 [00:00<00:00, 198.02it/s]


Loaded NB08: 51 maps


In [6]:
smaps['nb06'].peek()

### Align first image of each filter with each other.

In [7]:
# Make a MapSequence of the first image from each filter
temp = Map(
    [smaps[f"nb{i:02d}"][0] for i in range(1, 9)],
    sequence=True
)

# Create a dictionary mapping filter names to their indices
filter_index = {
    tmap.meta["ftr_name"]: i
    for i, tmap in enumerate(temp)
}

filter_index

{'NB06': 0,
 'NB07': 1,
 'NB08': 2,
 'NB01': 3,
 'NB04': 4,
 'NB03': 5,
 'NB02': 6,
 'NB05': 7}

In [9]:
fig, ax, roi = start_roi_selection(
    temp[filter_index["NB06"]]
)

In [10]:
template = create_submap(
    temp[filter_index["NB06"]],
    roi,
)

template.peek()

In [11]:
# Align all filter images to the NB03 image using the ROI template

index = filter_index["NB06"]

temp_2 = co_align(
    temp,
    layer_index=index,
    template=template
)

temp_2.peek()

In [12]:
# Replace the first map of each filter with its aligned version

for i in range(1, 9):
    nb = f"nb{i:02d}"

    filter_name = smaps[nb][1].meta["ftr_name"]
    aligned_index = filter_index[filter_name]

    smaps[nb].maps[0] = temp_2[aligned_index]

In [13]:
# Align the NB01 time sequence
coaligned_maps1 = align(smaps["nb06"])

Point 1: (273.8, 220.1)
Point 2: (418.9, 98.9)

ROI selected.
Run create_submap()


In [14]:
coaligned_maps2 = align(smaps["nb03"])

Point 1: (373.2, 505.8)
Point 2: (560.6, 360.6)

ROI selected.
Run create_submap()


In [15]:
coaligned_maps1.peek()

In [16]:
# Here we are taking a 60% contour during the peak of the flare in NB03 filter. In this case the peak is around 24th NB03 frame. 
# Feel free to choose any other frame as well but it will change the ligtcurves that you will get. 

j = 24
time = coaligned_maps2[j].meta['DHOBT_DT'].split('.')[0]
plt.figure()
contour_level = np.max(coaligned_maps2[j].data)*.60
contours = plt.contour(coaligned_maps2[j].data,levels=[contour_level],colors='red')
plt.imshow(coaligned_maps2[j].data,origin='lower')
plt.title(f'Frame {j}: with 60% contour in NB03 at {time}',fontweight='bold')
plt.show()

In [17]:
# creating a mask for the contour
path = contours.get_paths()[0]
data = coaligned_maps1[j].data
mask = np.zeros(data.shape, dtype=bool)
x, y = np.meshgrid(np.arange(data.shape[1]), np.arange(data.shape[0]))
points = np.vstack((x.flatten(), y.flatten())).T 
mask = path.contains_points(points).reshape(data.shape)
plt.figure()
plt.imshow(mask,origin='lower')
plt.title('Mask for 60% contour',fontweight='bold')
plt.show()

In [18]:
time1,lc1 = make_lc(mask,coaligned_maps1)
time2,lc2 = make_lc(mask,coaligned_maps2)

In [19]:
fig = plt.figure()
ax = fig.add_subplot()

ax.plot(time1, lc1, label='NB06')
ax.plot(time2, lc2, label='NB03')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.legend(loc='best', frameon=False)
fig.autofmt_xdate()
plt.show()

In [22]:
fig = plt.figure()
ax = fig.add_subplot()

ax.plot(time1, lc1, label='NB06')

ax2 = ax.twinx()
ax2.plot(time2, lc2, label='NB03', color='r')

ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))

# Combine legends from both axes
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax.legend(lines1 + lines2, labels1 + labels2,
          loc='best', frameon=False)

fig.autofmt_xdate()
plt.show()